In [24]:
import numpy as np
import pandas as pd
import argparse
import os
from itertools import combinations
from collections import defaultdict
from upsetplot import from_memberships, UpSet
from upsetplot import plot
import matplotlib.pyplot as plt
import logging
import csv


logging.basicConfig(
    filename='../../../../15th_20samples_output/script_log.txt',
    filemode='w',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)


logger = logging.getLogger()


In [25]:
def find_directories_excluding(path, exclude_name):
    directories = [
        d for d in os.listdir(path)
        if os.path.isdir(os.path.join(path, d)) and d != exclude_name
    ]
    return directories

# initialize dataFrames
def initialize_dfs(names):
    return {name: pd.DataFrame() for name in names}
    
# function to read csv files into dataframes
def read_csv_files(names, df_dict, base_path, columns):
    for name in names:
        file_path = os.path.join(base_path, name, "produce_final_form/final_form.csv")
        df_dict[name] = pd.read_csv(file_path, sep=',', usecols=columns)
        print(name)
        print(df_dict[name].head())

# def find_common_sequences(samples):
#     combos = []
#     counts = []
#     # common_sequences = []
#     # Iterate through all combinations of sample keys (categories)
#     for i in range(1, len(samples) + 1):
#         for combo in combinations(samples.keys(), i):
#             logger.info("combo", list(combo))
#             # Find common sequences and their count
#             common_seqs = set.intersection(*(set(samples[sample]) for sample in combo))
#             combos.append(list(combo))  # Store the combination as a list
#             counts.append(len(common_seqs))  # Store the count of common sequences
#             # common_sequences.append(list(common_seqs)) 
#             # print(common_sequences)

#     return combos, counts

def find_common_sequences(samples):
    combos = []
    counts = []
    for i in range(1, len(samples) + 1):
        for combo in combinations(samples.keys(), i):
            # Find common sequences and their count
            combos.append(list(combo))
            counts.append(len(set.intersection(*(set(samples[sample]()) for sample in combo))))
    return combos, counts
from itertools import combinations

def find_common_sequences_samples_more(samples):
    # Ensure all values in the samples dictionary are lists or sets
    if not all(isinstance(value, (list, set)) for value in samples.values()):
        raise ValueError("All values in 'samples' must be lists or sets.")
    result = {}
    num_samples = len(samples)
    total_unique_sequences = len(set(seq for sample in samples.values() for seq in sample))
    print("total_unique_sequence",total_unique_sequences)
    for i in range(2, num_samples + 1):  # Start from 2 since we're interested in >= 2 samples
        # Generate all combinations of `i` samples
        print("id",i)
        combos = combinations(samples.keys(), i)
        print("combination group number:",len(list(combinations(samples.keys(), i))))
        common_sequences = set()  # To store sequences occurring in at least `i` samples

        for combo in combos:
            # Find intersection of sequences for the current combination
            intersection = set.intersection(*(set(samples[sample]) for sample in combo))
            common_sequences.update(intersection)

        # Count and calculate the percentage
        count = len(common_sequences)
        # total_unique_sequences = len(set(seq for sample in samples.values() for seq in sample))
        percentage = (count / total_unique_sequences) * 100

        # Store results for current `i`
        result[f">= {i} samples"] = {"count": count, "percentage": f"{percentage:.2f}%" }
        print("result",result)
    return result


In [26]:
# define the base path for output csv files
# base_path = "../../../../15th_20samples_output"
base_path = "../../../../15th_20samples_output"
# names corresponding to the file names for bifi and ecoli
# bifi_names = [
#     "MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001", 
#     "MEF-BIEVs-1_S1_L001_R1_001", "MEF-BIEVs-2_S5_L001_R1_001", "MEF-BIEVs-3_S6_L001_R1_001"
# ]
# bifi_names = ["MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001"]
# # bifi_names = ["MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001"]
# ecoli_names = [
#     "MEE-OMVs-1_S4_L001_R1_001", "MEE-OMVs-2_S3_L001_R1_001", "MEE-OMVs-3_S12_L001_R1_001", 
#     "MEF-OMVs-1_S2_L001_R1_001", "MEF-OMVs-2_S11_L001_R1_001", "MEF-OMVs-3_S10_L001_R1_001"
# ]
# all_names = [   "MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001", 
#                 "MEF-BIEVs-1_S1_L001_R1_001", "MEF-BIEVs-2_S5_L001_R1_001", "MEF-BIEVs-3_S6_L001_R1_001",
#                 "MEE-OMVs-1_S4_L001_R1_001", "MEE-OMVs-2_S3_L001_R1_001", "MEE-OMVs-3_S12_L001_R1_001", 
#                 "MEF-OMVs-1_S2_L001_R1_001", "MEF-OMVs-2_S11_L001_R1_001", "MEF-OMVs-3_S10_L001_R1_001" ]
# all_names = ["SRR12412943","SRR12412944"]
# all_names = [
#         "MEE-OMVs-1_S4_L001_R1_001", "MEE-OMVs-2_S3_L001_R1_001", "MEE-OMVs-3_S12_L001_R1_001", 
#     "MEF-OMVs-1_S2_L001_R1_001", "MEF-OMVs-2_S11_L001_R1_001", "MEF-OMVs-3_S10_L001_R1_001"
# ]
#all_names = ["SRR20723831","SRR20723832","SRR20723833"]
# all_names = ["SRR25896993_1","SRR25896994_1","SRR25896995_1"]
# all_names = ["SRR12412943","SRR12412944"]
all_names = find_directories_excluding(base_path, "total_samples_analysis")
print("all_names",all_names)
# all_names = ["SRR1638859","SRR1638860"]
# columns to read from the original CSV files
read_columns = ['sequence', 'mirBase', 'linearfold', 'mirdeep2', 'representative_id','qseqid_count','same_seq_ids']

# df_bifi = initialize_dfs(bifi_names)
# df_ecoli = initialize_dfs(ecoli_names)
df_all = initialize_dfs(all_names)
logger.info("1")
#logger.info(df_all)
# read_csv_files(bifi_names, df_bifi, base_path, read_columns)
# read_csv_files(ecoli_names, df_ecoli, base_path, read_columns)
read_csv_files(all_names, df_all, base_path, read_columns)
logger.info("2")
# logger.info(df_all)
# i = 1
# for name, df in df_all.items():
#     logger.info(f"DataFrame for {name}:")
#     if 'sequence' in df.columns:
#         logger.info(df['sequence'])  # Check if sequence column has data
#     else:
#         logger.info("No 'sequence' column found in this DataFrame.")
# bifi_sequences = {name: df['sequence'].tolist() for name, df in df_bifi.items()}
# ecoli_sequences = {name: df['sequence'].tolist() for name, df in df_ecoli.items()}
#all_sequences = {name: df_single['sequence'].tolist() for name,df_single in df_all.items()}
###avoid too huge memory, change list to generator
# all_sequences = {name: (seq for seq in df['sequence']) for name, df in df_all.items()}
### also needs a special generator, like reiterated generator,for upset plot
#all_sequences = {name: (lambda df=df: (seq for seq in df['sequence'])) for name, df in df_all.items()}
###below this use for the find_common functions 
all_sequences = {name: list(seq for seq in df['sequence']) for name, df in df_all.items()}

logger.info("3")
# for i, (key, value) in enumerate(all_sequences.items()):
#     if i >= 3:  # Limit to the first 3 items
#         break
#     logger.info(f"{key}: {value}")
# print("bifi_sequences", bifi_sequences)

# combos_bifi, counts_bifi= find_common_sequences(bifi_sequences)
# print("combos_bifi", combos_bifi)
# print("counts_bifi", counts_bifi)
if len(all_names) >= 9:
    results = find_common_sequences_samples_more(all_sequences)

    df = pd.DataFrame.from_dict(results, orient="index")
    df.to_csv("../../../../15th_20samples_output/total_samples_analysis/common_sequences_occur_rates.csv")
else:
    # combos_ecoli, counts_ecoli  = find_common_sequences(ecoli_sequences)
    combos_all, counts_all  = find_common_sequences(all_sequences)
    print(combos_all)
    print(counts_all)
    logger.info("4")
    example = from_memberships(
                combos_all,
                counts_all
            )
    logger.info(example)
    plot(example)  

    # fig = plt.figure(figsize=(10, 6), dpi=30)  # Adjust figsize and dpi as needed


    # upset = UpSet(example)
    # upset.plot(fig=fig)


    # plt.savefig("../../../../15th_20samples_output/total_samples_analysis/all_upset_plot.pdf", format='pdf', bbox_inches='tight')
    plt.savefig("../../../../15th_20samples_output/total_samples_analysis/all_upset_plot.png", format='png', bbox_inches='tight')
    plt.close()
        # logger.info("3")

all_names ['ERR1396767', 'ERR1396755', 'ERR1296699', 'ERR1296700', 'ERR1396758', 'ERR1396760', 'ERR1396764', 'ERR1396756', 'ERR1296704', 'ERR1296703', 'ERR1396763', 'ERR1396759', 'ERR1296701', 'ERR1396761', 'ERR1396766', 'ERR1396754', 'ERR1296702', 'ERR1396762', 'ERR1396765', 'ERR1396757']
ERR1396767
                            sequence  mirBase  linearfold  mirdeep2  \
0  GCCGGTGTAGCTCAGTCGGTAGAGCAGCGCATA        1           1         0   
1                TTGATGAACGGCGGCGGGA        0           1         0   
2           CTACGAACCAGAGGGTCGGGCGTT        0           1         0   
3            AGGACCGGAGTGGACGAACCTCT        0           1         0   
4               GCGGACGTGGCGAAATTGGT        0           1         0   

   representative_id  qseqid_count  \
0                  2         18555   
1                  3             2   
2                  4             4   
3                  6            67   
4                  8         26184   

                                        sa

KeyboardInterrupt: 

In [ ]:

# example = from_memberships(
#    combos_bifi,
#    counts_bifi
# )
# plot(example)  
# plt.savefig("../../../../12th_9samples_output/total_samples_analysis/bifi_upset_plot.png", format='png', bbox_inches='tight')
# plt.close()


In [ ]:
# example = from_memberships(
#    combos_ecoli,
#    counts_ecoli
# )
# plot(example)  
# plt.savefig("../../../../12th_9samples_output/total_samples_analysis/ecoli_upset_plot.png", format='png', bbox_inches='tight')
# plt.close()

In [ ]:
# example = from_memberships(
#    combos_all,
#    counts_all
# )
# logger.info(example)
# plot(example)  

# # fig = plt.figure(figsize=(10, 6), dpi=30)  # Adjust figsize and dpi as needed


# # upset = UpSet(example)
# # upset.plot(fig=fig)


# # plt.savefig("../../../../12th_9samples_output/total_samples_analysis/all_upset_plot.pdf", format='pdf', bbox_inches='tight')
# plt.savefig("../../../../12th_9samples_output/total_samples_analysis/all_upset_plot.png", format='png', bbox_inches='tight')
# plt.close()

In [ ]:
# list1 = ["ATCG", "GCTA", "TTAA", "CCGG"]
# list2 = ["GCTA", "GCTA", "AATT", "GGCC"]


# set1 = set(list1)
# set2 = set(list2)
# #print(set2)
# # Find common sequences
# common_sequences = set1.intersection(set2)
# print(common_sequences)

